# COMP9517 - Computer Vision - Group Project Demo

## Setting up the Environment

In [43]:
%%bash
#!/bin/bash

if command -v uv 2> /dev/null
then
    echo Running uv sync...
    uv sync
    uv pip install git+https://github.com/lucasb-eyer/pydensecrf.git
else
    echo "Please install uv to setup the environment"
    echo "Hint: Run 'pip install uv'"
fi

printf "\nRunning in Directory: %s" "$(pwd)"


/opt/homebrew/bin/uv
Running uv sync...


Resolved 54 packages in 9ms
Uninstalled 1 package in 1ms
 - pydensecrf==1.0 (from git+https://github.com/lucasb-eyer/pydensecrf.git@2723c7fa4f2ead16ae1ce3d8afe977724bb8f87f)
Using Python 3.12.12 environment at: /Users/junhao_bai/Workspace/unsw/COMP9517_26T1/cs9517_group_project/.venv
Resolved 1 package in 386ms
Installed 1 package in 2ms
 + pydensecrf==1.0 (from git+https://github.com/lucasb-eyer/pydensecrf.git@2723c7fa4f2ead16ae1ce3d8afe977724bb8f87f)



Running in Directory: /Users/junhao_bai/Workspace/unsw/COMP9517_26T1/cs9517_group_project/notebooks

## Classic CV methods

In [44]:
%%bash
unset MPLBACKEND
source ../.venv/bin/activate

jq -r 'keys[]' ../configs/traditional_cv.json | while read -r method; do
    # Run and Evaluate traditional cv algorithms
    cmd="../scripts/classic_cv.py -R $method -C traditional_cv.json"
    echo "Running Command $cmd"
    eval "$cmd"

    # Plot cross-level comparison
    cmd="../scripts/compare.py -m robustness -D test -R $method"
    echo "Running Command $cmd"
    eval "$cmd"
done

# Compare all traditional cv methods
cmd="../scripts/compare.py -m cross -D test --method cv -C traditional_cv.json"
echo Running Command "$cmd"
eval "$cmd"

Running Command ../scripts/classic_cv.py -R crf_method -C traditional_cv.json
Running Command ../scripts/compare.py -m robustness -D test -R crf_method
Running Command ../scripts/classic_cv.py -R edge_method -C traditional_cv.json
Running Command ../scripts/compare.py -m robustness -D test -R edge_method
Running Command ../scripts/classic_cv.py -R excessive_green_method -C traditional_cv.json
Running Command ../scripts/compare.py -m robustness -D test -R excessive_green_method
Running Command ../scripts/classic_cv.py -R grabcut_method -C traditional_cv.json
Running Command ../scripts/compare.py -m robustness -D test -R grabcut_method
Running Command ../scripts/classic_cv.py -R hsv_segmentation_method -C traditional_cv.json
Running Command ../scripts/compare.py -m robustness -D test -R hsv_segmentation_method
Running Command ../scripts/classic_cv.py -R kmeans_method -C traditional_cv.json
Running Command ../scripts/compare.py -m robustness -D test -R kmeans_method
Running Command ../scr

## Machine Learning methods

In [45]:
%%bash
unset MPLBACKEND
source ../.venv/bin/activate

capitalize() {
    tr '[:lower:]' '[:upper:]'
}

channel_modes=("rgb" "hsv" "exg")
unset mode

for m in "${channel_modes[@]}"; do
    mode="$mode""$m"
    run_name=$(echo "$mode" | capitalize)

    # Train random forest model
    cmd="python3 -m project.machine_learning.train_rf -R RF_$run_name -fm $mode"
    echo Running Command "$cmd"
    eval "$cmd"

    # Evaluate random forest model
    cmd="python3 -m project.machine_learning.evaluate_rf -R RF_$run_name -m test -fm $mode"
    echo Running Command "$cmd"
    eval "$cmd"

    # Train logistic regression model
    cmd="python3 -m project.machine_learning.train_lr -R LR_$run_name -fm $mode"
    echo Running Command "$cmd"
    eval "$cmd"

    # Evaluate logistic regression model
    cmd="python3 -m project.machine_learning.evaluate_lr -R LR_$run_name -m test -fm $mode"
    echo Running Command "$cmd"
    eval "$cmd"

    # Concat channels
    mode="$mode"_
done

unset mode

# Compare between classifiers
cmd="../scripts/plot_classifier_comparison.py"
echo Running Command "$cmd"
eval "$cmd"

# Compare within the random forests
cmd="../scripts/plot_rf_feature_comparison.py"
echo Running Command "$cmd"
eval "$cmd"

Running Command python3 -m project.machine_learning.train_rf -R RF_RGB -fm rgb
[OK] Trained RF run=RF_RGB
Train time: 3.35s
Validation inference time: 3.40s
Validation accuracy: 0.8767036038653582
Running Command python3 -m project.machine_learning.evaluate_rf -R RF_RGB -m test -fm rgb
[OK] Evaluated RF run=RF_RGB, mode=test
Inference time: 3.45s
Accuracy: 0.8652669943397039
Running Command python3 -m project.machine_learning.train_lr -R LR_RGB -fm rgb
[OK] Trained LR run=LR_RGB
Train time: 0.16s
Validation inference time: 0.02s
Validation accuracy: 0.880401022834883
Running Command python3 -m project.machine_learning.evaluate_lr -R LR_RGB -m test -fm rgb
[OK] Evaluated LR run=LR_RGB, mode=test
Inference time: 0.02s
Accuracy: 0.8697284456783747
Running Command python3 -m project.machine_learning.train_rf -R RF_RGB_HSV -fm rgb_hsv
[OK] Trained RF run=RF_RGB_HSV
Train time: 6.36s
Validation inference time: 2.73s
Validation accuracy: 0.8743529937155647
Running Command python3 -m project.m

## Deep learning methods

In [46]:
%%bash
unset MPLBACKEND
source ../.venv/bin/activate

# resume=last.pt

jq -r 'keys[]' ../configs/networks.json | while read -r run; do
    # Train the neural network
    cmd="python3 -m project.deep_learning.train_neural_network -R $run -C networks.json"

    if [ -n "$resume" ]; then
        cmd="$cmd -r $resume"
    fi

    echo "Running Command $cmd"
    eval "$cmd"

    # Evaluate the neural network
    cmd="python3 -m project.deep_learning.evaluate_neural_network -R $run -C networks.json -m test"
    echo "Running Command $cmd"
    eval "$cmd"
    # Compare between robustness levels
    cmd="../scripts/compare.py -m robustness -D test -R $run"
    echo "Running Command $cmd"
    eval "$cmd"
done

# Compare all deep learning models
cmd="../scripts/compare.py -m cross -D test --method dl -C networks.json"
echo Running Command "$cmd"
eval "$cmd"

Running Command python3 -m project.deep_learning.train_neural_network -R ASPPResUnet -C networks.json


2026-04-15 19:12:39,818 [INFO] train logger: epoch 1 | train loss: 0.376572 | val loss: 0.209560
2026-04-15 19:13:13,495 [INFO] train logger: epoch 2 | train loss: 0.233318 | val loss: 0.205194
2026-04-15 19:13:47,144 [INFO] train logger: epoch 3 | train loss: 0.203755 | val loss: 0.168315
2026-04-15 19:14:20,756 [INFO] train logger: epoch 4 | train loss: 0.188882 | val loss: 0.181906
2026-04-15 19:14:53,397 [INFO] train logger: epoch 5 | train loss: 0.189076 | val loss: 0.152316
2026-04-15 19:15:26,906 [INFO] train logger: epoch 6 | train loss: 0.179060 | val loss: 0.160307
2026-04-15 19:15:59,473 [INFO] train logger: epoch 7 | train loss: 0.174762 | val loss: 0.165559
2026-04-15 19:16:32,079 [INFO] train logger: epoch 8 | train loss: 0.176485 | val loss: 0.169576
2026-04-15 19:17:04,718 [INFO] train logger: epoch 9 | train loss: 0.166785 | val loss: 0.149814
2026-04-15 19:17:38,375 [INFO] train logger: epoch 10 | train loss: 0.163873 | val loss: 0.153097
2026-04-15 19:18:11,009 [INFO

Running Command python3 -m project.deep_learning.evaluate_neural_network -R ASPPResUnet -C networks.json -m test
Running Command ../scripts/compare.py -m robustness -D test -R ASPPResUnet
Running Command python3 -m project.deep_learning.train_neural_network -R AttentionASPPResUnet -C networks.json


2026-04-15 19:38:12,462 [INFO] train logger: epoch 1 | train loss: 0.369141 | val loss: 0.202493
2026-04-15 19:38:46,686 [INFO] train logger: epoch 2 | train loss: 0.223612 | val loss: 0.216807
2026-04-15 19:39:20,066 [INFO] train logger: epoch 3 | train loss: 0.199703 | val loss: 0.163672
2026-04-15 19:39:54,269 [INFO] train logger: epoch 4 | train loss: 0.189229 | val loss: 0.174437
2026-04-15 19:40:27,586 [INFO] train logger: epoch 5 | train loss: 0.187257 | val loss: 0.154280
2026-04-15 19:41:01,807 [INFO] train logger: epoch 6 | train loss: 0.182566 | val loss: 0.159278
2026-04-15 19:41:35,311 [INFO] train logger: epoch 7 | train loss: 0.176559 | val loss: 0.173438
2026-04-15 19:42:08,610 [INFO] train logger: epoch 8 | train loss: 0.172669 | val loss: 0.159888
2026-04-15 19:42:41,894 [INFO] train logger: epoch 9 | train loss: 0.163432 | val loss: 0.151199
2026-04-15 19:43:16,066 [INFO] train logger: epoch 10 | train loss: 0.160681 | val loss: 0.156559
2026-04-15 19:43:49,371 [INFO

Running Command python3 -m project.deep_learning.evaluate_neural_network -R AttentionASPPResUnet -C networks.json -m test
Running Command ../scripts/compare.py -m robustness -D test -R AttentionASPPResUnet
Running Command python3 -m project.deep_learning.train_neural_network -R ResUnet -C networks.json


2026-04-15 20:00:13,138 [INFO] train logger: epoch 1 | train loss: 0.408097 | val loss: 0.228774
2026-04-15 20:00:39,677 [INFO] train logger: epoch 2 | train loss: 0.234801 | val loss: 0.234808
2026-04-15 20:01:05,701 [INFO] train logger: epoch 3 | train loss: 0.205198 | val loss: 0.165031
2026-04-15 20:01:32,202 [INFO] train logger: epoch 4 | train loss: 0.187695 | val loss: 0.171183
2026-04-15 20:01:58,226 [INFO] train logger: epoch 5 | train loss: 0.185739 | val loss: 0.152358
2026-04-15 20:02:24,722 [INFO] train logger: epoch 6 | train loss: 0.178442 | val loss: 0.158937
2026-04-15 20:02:50,770 [INFO] train logger: epoch 7 | train loss: 0.172331 | val loss: 0.176955
2026-04-15 20:03:16,889 [INFO] train logger: epoch 8 | train loss: 0.173382 | val loss: 0.160590
2026-04-15 20:03:42,893 [INFO] train logger: epoch 9 | train loss: 0.163564 | val loss: 0.147633
2026-04-15 20:04:09,380 [INFO] train logger: epoch 10 | train loss: 0.161524 | val loss: 0.153567
2026-04-15 20:04:35,456 [INFO

Running Command python3 -m project.deep_learning.evaluate_neural_network -R ResUnet -C networks.json -m test
Running Command ../scripts/compare.py -m robustness -D test -R ResUnet
Running Command python3 -m project.deep_learning.train_neural_network -R UNet_Baseline -C networks.json


2026-04-15 20:19:56,321 [INFO] train logger: epoch 1 | train loss: 0.669059 | val loss: 0.637237
2026-04-15 20:20:13,475 [INFO] train logger: epoch 2 | train loss: 0.622445 | val loss: 0.521665
2026-04-15 20:20:30,606 [INFO] train logger: epoch 3 | train loss: 0.478769 | val loss: 0.417242
2026-04-15 20:20:47,728 [INFO] train logger: epoch 4 | train loss: 0.363751 | val loss: 0.251942
2026-04-15 20:21:04,867 [INFO] train logger: epoch 5 | train loss: 0.286823 | val loss: 0.280349
2026-04-15 20:21:21,601 [INFO] train logger: epoch 6 | train loss: 0.281956 | val loss: 0.240734
2026-04-15 20:21:38,717 [INFO] train logger: epoch 7 | train loss: 0.259017 | val loss: 0.209142
2026-04-15 20:21:55,931 [INFO] train logger: epoch 8 | train loss: 0.246678 | val loss: 0.189841
2026-04-15 20:22:13,062 [INFO] train logger: epoch 9 | train loss: 0.236896 | val loss: 0.195143
2026-04-15 20:22:29,818 [INFO] train logger: epoch 10 | train loss: 0.229479 | val loss: 0.181148
2026-04-15 20:22:46,992 [INFO

Running Command python3 -m project.deep_learning.evaluate_neural_network -R UNet_Baseline -C networks.json -m test
Running Command ../scripts/compare.py -m robustness -D test -R UNet_Baseline
Running Command ../scripts/compare.py -m cross -D test --method dl -C networks.json
